In [1]:
pip install SPARQLWrapper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.1/528.1 kB 21.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 4.2 MB/s eta 0:00:00


5 Output with only enriched with director column

In [2]:
import pandas as pd
from SPARQLWrapper import SPARQLWrapper, JSON

def get_director_from_dbpedia(dbpedia_uri):
    query = f'''
        PREFIX dbo: <http://dbpedia.org/ontology/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        SELECT DISTINCT ?directorName
        WHERE {{
            <{dbpedia_uri}> dbo:director ?director .
            ?director rdfs:label ?directorName .
            FILTER (LANG(?directorName) = 'en')
        }}
    '''

    sparql = SPARQLWrapper('http://dbpedia.org/sparql')
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    bindings = results['results']['bindings']

    if bindings:
        return bindings[0]['directorName']['value']
    else:
        return None

# Load the MovieLens 1M dataset
movies_file = 'movies.dat'  # Replace with the actual path to your movies file
movies_data = pd.read_csv(movies_file, sep='::', engine='python', names=['MovieID', 'Title', 'Genres'], encoding='latin1')

# Load the mapping file that contains the DBpedia URIs for each MovieLens movie title
mapping_file = 'MappingMovielens2DBpedia-1.2.tsv'  # Replace with the actual path to your mapping file
mapping_data = pd.read_csv(mapping_file, sep='\t', names=['MovieLensTitle', 'DBpediaURI'], encoding='utf-8')

# Create empty columns for directors and DBpedia URIs
movies_data['Director'] = ""
movies_data['DBpediaURI'] = ""

# Counter for the number of mappings found
num_mappings_found = 0

# Enrich the MovieLens dataset with director information and DBpedia URIs
for idx, row in movies_data.iterrows():
    movie_title = row['Title']

    # Check if the movie title exists in the mapping dictionary
    if movie_title in mapping_data['MovieLensTitle'].values:
        # Retrieve the corresponding DBpedia URI
        dbpedia_uri = mapping_data.loc[mapping_data['MovieLensTitle'] == movie_title, 'DBpediaURI'].iloc[0]
        # Get the director name from DBpedia
        director = get_director_from_dbpedia(dbpedia_uri)

        if director is not None:
            # Assign the director and DBpedia URI to the movie
            movies_data.at[idx, 'Director'] = director
            movies_data.at[idx, 'DBpediaURI'] = dbpedia_uri
            # Increment the counter
            num_mappings_found += 1
            # Check if the desired number of mappings have been found
            if num_mappings_found == 5:
                break
        else:
            print(f"No director found for movie title: {movie_title}")

# Write the enriched dataset to a CSV file
enriched_data_file = 'movies_enriched.csv'  # Replace with the desired output file path
movies_data.to_csv(enriched_data_file, index=False)

Run for all data for only director

In [ ]:
import pandas as pd
from SPARQLWrapper import SPARQLWrapper, JSON

def get_director_from_dbpedia(dbpedia_uri):
    query = f'''
        PREFIX dbo: <http://dbpedia.org/ontology/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        SELECT DISTINCT ?directorName
        WHERE {{
            <{dbpedia_uri}> dbo:director ?director .
            ?director rdfs:label ?directorName .
            FILTER (LANG(?directorName) = 'en')
        }}
    '''

    sparql = SPARQLWrapper('http://dbpedia.org/sparql')
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    bindings = results['results']['bindings']

    if bindings:
        return bindings[0]['directorName']['value']
    else:
        return None

# Load the MovieLens 1M dataset
movies_file = 'movies.dat'  # Replace with the actual path to your movies file
movies_data = pd.read_csv(movies_file, sep='::', engine='python', names=['MovieID', 'Title', 'Genres'], encoding='latin1')

# Load the mapping file that contains the DBpedia URIs for each MovieLens movie title
mapping_file = 'MappingMovielens2DBpedia-1.2.tsv'  # Replace with the actual path to your mapping file
mapping_data = pd.read_csv(mapping_file, sep='\t', names=['MovieLensTitle', 'DBpediaURI'], encoding='utf-8')

# Create empty columns for directors and DBpedia URIs
movies_data['Director'] = ""
movies_data['DBpediaURI'] = ""

# Enrich the MovieLens dataset with director information and DBpedia URIs
for idx, row in movies_data.iterrows():
    movie_title = row['Title']

    # Check if the movie title exists in the mapping dictionary
    if movie_title in mapping_data['MovieLensTitle'].values:
        # Retrieve the corresponding DBpedia URI
        dbpedia_uri = mapping_data.loc[mapping_data['MovieLensTitle'] == movie_title, 'DBpediaURI'].iloc[0]
        # Get the director name from DBpedia
        director = get_director_from_dbpedia(dbpedia_uri)

        if director is not None:
            # Assign the director and DBpedia URI to the movie
            movies_data.at[idx, 'Director'] = director
            movies_data.at[idx, 'DBpediaURI'] = dbpedia_uri
        else:
            print(f"No director found for movie title: {movie_title}")
    else:
        print(f"No mapping found for movie title: {movie_title}")

# Write the enriched dataset to a CSV file
enriched_data_file = 'movies_enriched.csv'  # Replace with the desired output file path
movies_data.to_csv(enriched_data_file, index=False)


No mapping found for movie title: Toy Story (1995)
No director found for movie title: Tom and Huck (1995)
No director found for movie title: Sudden Death (1995)
No director found for movie title: Casino (1995)
No director found for movie title: Money Train (1995)
No director found for movie title: Assassins (1995)
No director found for movie title: Now and Then (1995)
No mapping found for movie title: Across the Sea of Time (1995)
No director found for movie title: Clueless (1995)
No director found for movie title: Mortal Kombat (1995)
No director found for movie title: Usual Suspects, The (1995)
No mapping found for movie title: Guardian Angel (1994)
No mapping found for movie title: Kids of the Round Table (1995)
No director found for movie title: Home for the Holidays (1995)
No mapping found for movie title: Postino, Il (The Postman) (1994)
No mapping found for movie title: Confessional, The (Le Confessionnal) (1995)
No director found for movie title: Fair Game (1995)
No mapping fou

KeyboardInterrupt: ignored

[5 outputs for test] Certainly! Here's the updated code that extracts directors who have directed at least 3 movies and actors who have starred in at least 3 movies. The code will stop after finding 5 such mappings:


In [3]:
import pandas as pd
from SPARQLWrapper import SPARQLWrapper, JSON

def get_director_from_dbpedia(dbpedia_uri):
    query = f'''
        PREFIX dbo: <http://dbpedia.org/ontology/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        SELECT DISTINCT ?directorName
        WHERE {{
            <{dbpedia_uri}> dbo:director ?director .
            ?director rdfs:label ?directorName .
            FILTER (LANG(?directorName) = 'en')
        }}
    '''

    sparql = SPARQLWrapper('http://dbpedia.org/sparql')
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    bindings = results['results']['bindings']

    if bindings:
        return bindings[0]['directorName']['value']
    else:
        return None

def get_starring_actors_from_dbpedia(dbpedia_uri):
    query = f'''
        PREFIX dbo: <http://dbpedia.org/ontology/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        SELECT DISTINCT ?actorName
        WHERE {{
            <{dbpedia_uri}> dbo:starring ?actor .
            ?actor rdfs:label ?actorName .
            FILTER (LANG(?actorName) = 'en')
        }}
    '''

    sparql = SPARQLWrapper('http://dbpedia.org/sparql')
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    bindings = results['results']['bindings']

    actors = []
    for binding in bindings:
        actors.append(binding['actorName']['value'])

    return actors

# Load the MovieLens 1M dataset
movies_file = 'movies.dat'  # Replace with the actual path to your movies file
movies_data = pd.read_csv(movies_file, sep='::', engine='python', names=['MovieID', 'Title', 'Genres'], encoding='latin1')

# Load the mapping file that contains the DBpedia URIs for each MovieLens movie title
mapping_file = 'MappingMovielens2DBpedia-1.2.tsv'  # Replace with the actual path to your mapping file
mapping_data = pd.read_csv(mapping_file, sep='\t', names=['MovieLensTitle', 'DBpediaURI'], encoding='utf-8')

# Create empty columns for directors and DBpedia URIs
movies_data['Director'] = ""
movies_data['Starring'] = ""
movies_data['DBpediaURI'] = ""

# Counters for the number of mappings found
num_directors_found = 0
num_actors_found = 0

# Enrich the MovieLens dataset with director information, starring actors, and DBpedia URIs
for idx, row in movies_data.iterrows():
    movie_title = row['Title']

    # Check if the movie title exists in the mapping dictionary
    if movie_title in mapping_data['MovieLensTitle'].values:
        # Retrieve the corresponding DBpedia URI
        dbpedia_uri = mapping_data.loc[mapping_data['MovieLensTitle'] == movie_title, 'DBpediaURI'].iloc[0]
        # Get the director name from DBpedia
        director = get_director_from_dbpedia(dbpedia_uri)
        # Get the starring actors from DBpedia
        actors = get_starring_actors_from_dbpedia(dbpedia_uri)

        if director is not None and len(actors) >= 3:
            # Assign the director, starring actors, and DBpedia URI to the movie
            movies_data.at[idx, 'Director'] = director
            movies_data.at[idx, 'Starring'] = ', '.join(actors)
            movies_data.at[idx, 'DBpediaURI'] = dbpedia_uri
            # Increment the counters
            num_directors_found += 1
            num_actors_found += 1
            # Check if the desired number of mappings have been found
            if num_directors_found == 5 and num_actors_found == 5:
                break
        else:
            print(f"No director found for movie title: {movie_title} or insufficient actors found")

# Write the enriched dataset to a CSV file
enriched_data_file = 'movies_enriched.csv'  # Replace with the desired output file path
movies_data.to_csv(enriched_data_file, index=False)

[ALL- need long time to process] Certainly! To extract the directors who have directed at least 3 movies and the actors who have starred in at least 3 movies for all rows in the MovieLens dataset, you can update the code as follows:



In [4]:
import pandas as pd
from SPARQLWrapper import SPARQLWrapper, JSON

def get_director_from_dbpedia(dbpedia_uri):
    query = f'''
        PREFIX dbo: <http://dbpedia.org/ontology/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        SELECT DISTINCT ?directorName
        WHERE {{
            <{dbpedia_uri}> dbo:director ?director .
            ?director rdfs:label ?directorName .
            FILTER (LANG(?directorName) = 'en')
        }}
    '''

    sparql = SPARQLWrapper('http://dbpedia.org/sparql')
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    bindings = results['results']['bindings']

    if bindings:
        return bindings[0]['directorName']['value']
    else:
        return None

def get_starring_actors_from_dbpedia(dbpedia_uri):
    query = f'''
        PREFIX dbo: <http://dbpedia.org/ontology/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        SELECT DISTINCT ?actorName
        WHERE {{
            <{dbpedia_uri}> dbo:starring ?actor .
            ?actor rdfs:label ?actorName .
            FILTER (LANG(?actorName) = 'en')
        }}
    '''

    sparql = SPARQLWrapper('http://dbpedia.org/sparql')
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    bindings = results['results']['bindings']

    actors = []
    for binding in bindings:
        actors.append(binding['actorName']['value'])

    return actors

# Load the MovieLens 1M dataset
movies_file = 'movies.dat'  # Replace with the actual path to your movies file
movies_data = pd.read_csv(movies_file, sep='::', engine='python', names=['MovieID', 'Title', 'Genres'], encoding='latin1')

# Load the mapping file that contains the DBpedia URIs for each MovieLens movie title
mapping_file = 'MappingMovielens2DBpedia-1.2.tsv'  # Replace with the actual path to your mapping file
mapping_data = pd.read_csv(mapping_file, sep='\t', names=['MovieLensTitle', 'DBpediaURI'], encoding='utf-8')

# Create empty columns for directors and DBpedia URIs
movies_data['Director'] = ""
movies_data['Starring'] = ""
movies_data['DBpediaURI'] = ""

# Enrich the MovieLens dataset with director information, starring actors, and DBpedia URIs
for idx, row in movies_data.iterrows():
    movie_title = row['Title']

    # Check if the movie title exists in the mapping dictionary
    if movie_title in mapping_data['MovieLensTitle'].values:
        # Retrieve the corresponding DBpedia URI
        dbpedia_uri = mapping_data.loc[mapping_data['MovieLensTitle'] == movie_title, 'DBpediaURI'].iloc[0]
        # Get the director name from DBpedia
        director = get_director_from_dbpedia(dbpedia_uri)
        # Get the starring actors from DBpedia
        actors = get_starring_actors_from_dbpedia(dbpedia_uri)

        if director is not None and len(actors) >= 3:
            # Assign the director, starring actors, and DBpedia URI to the movie
            movies_data.at[idx, 'Director'] = director
            movies_data.at[idx, 'Starring'] = ', '.join(actors)
            movies_data.at[idx, 'DBpediaURI'] = dbpedia_uri
        else:
            print(f"No director found for movie title: {movie_title} or insufficient actors found")

# Write the enriched dataset to a CSV file
enriched_data_file = 'movies_enriched.csv'  # Replace with the desired output file path
movies_data.to_csv(enriched_data_file, index=False)

No director found for movie title: Tom and Huck (1995) or insufficient actors found
No director found for movie title: Sudden Death (1995) or insufficient actors found
No director found for movie title: Casino (1995) or insufficient actors found
No director found for movie title: Money Train (1995) or insufficient actors found
No director found for movie title: Assassins (1995) or insufficient actors found
No director found for movie title: Now and Then (1995) or insufficient actors found
No director found for movie title: Persuasion (1995) or insufficient actors found
No director found for movie title: Dangerous Minds (1995) or insufficient actors found
No director found for movie title: Babe (1995) or insufficient actors found
No director found for movie title: Clueless (1995) or insufficient actors found
No director found for movie title: Mortal Kombat (1995) or insufficient actors found
No director found for movie title: Seven (Se7en) (1995) or insufficient actors found
No director

KeyboardInterrupt: ignored

Apply parallel processing

20 output for test

In [8]:
import pandas as pd
from SPARQLWrapper import SPARQLWrapper, JSON
from multiprocessing import Pool
import time

def get_director_from_dbpedia(movie_data):
    movie_title, dbpedia_uri = movie_data
    query = f'''
        PREFIX dbo: <http://dbpedia.org/ontology/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        SELECT DISTINCT ?directorName
        WHERE {{
            <{dbpedia_uri}> dbo:director ?director .
            ?director rdfs:label ?directorName .
            FILTER (LANG(?directorName) = 'en')
        }}
    '''

    sparql = SPARQLWrapper('http://dbpedia.org/sparql')
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    bindings = results['results']['bindings']

    if bindings:
        return movie_title, bindings[0]['directorName']['value']
    else:
        return movie_title, None

def get_starring_actors_from_dbpedia(movie_data):
    movie_title, dbpedia_uri = movie_data
    query = f'''
        PREFIX dbo: <http://dbpedia.org/ontology/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        SELECT DISTINCT ?actorName
        WHERE {{
            <{dbpedia_uri}> dbo:starring ?actor .
            ?actor rdfs:label ?actorName .
            FILTER (LANG(?actorName) = 'en')
        }}
    '''

    sparql = SPARQLWrapper('http://dbpedia.org/sparql')
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    bindings = results['results']['bindings']

    actors = []
    for binding in bindings:
        actors.append(binding['actorName']['value'])

    return movie_title, actors

# Load the MovieLens 1M dataset
movies_file = 'movies.dat'  # Replace with the actual path to your movies file
movies_data = pd.read_csv(movies_file, sep='::', engine='python', names=['MovieID', 'Title', 'Genres'], encoding='latin1')

# Load the mapping file that contains the DBpedia URIs for each MovieLens movie title
mapping_file = 'MappingMovielens2DBpedia-1.2.tsv'  # Replace with the actual path to your mapping file
mapping_data = pd.read_csv(mapping_file, sep='\t', names=['MovieLensTitle', 'DBpediaURI'], encoding='utf-8')

# Create empty columns for directors, starring actors, and DBpedia URIs
movies_data['Director'] = ""
movies_data['Starring'] = ""
movies_data['DBpediaURI'] = ""

# Define the number of worker processes to use
num_processes = 4

# Create a pool of worker processes
pool = Pool(processes=num_processes)

# Enrich the MovieLens dataset with director information, starring actors, and DBpedia URIs
enriched_results = []
for idx, row in movies_data.iterrows():
    movie_title = row['Title']

    # Check if the movie title exists in the mapping dictionary
    if movie_title in mapping_data['MovieLensTitle'].values:
        # Retrieve the corresponding DBpedia URI
        dbpedia_uri = mapping_data.loc[mapping_data['MovieLensTitle'] == movie_title, 'DBpediaURI'].iloc[0]
        # Add the movie data to the list for processing
        enriched_results.append((movie_title, dbpedia_uri))

        # Stop if 20 results are obtained
        if len(enriched_results) >= 20:
            break

# Use parallel processing to retrieve director and actor information in batches
batch_size = 10
num_batches = len(enriched_results) // batch_size + 1

start_time = time.time()

for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = (i + 1) * batch_size
    batch_data = enriched_results[start_idx:end_idx]

    # Use parallel processing to retrieve director and actor information
    director_results = pool.map(get_director_from_dbpedia, batch_data)
    actor_results = pool.map(get_starring_actors_from_dbpedia, batch_data)

    # Update the MovieLens dataset with the retrieved information
    for movie_title, director_name in director_results:
        if director_name is not None:
            movies_data.loc[movies_data['Title'] == movie_title, 'Director'] = director_name

    for movie_title, actor_names in actor_results:
        if len(actor_names) >= 3:
            movies_data.loc[movies_data['Title'] == movie_title, 'Starring'] = ', '.join(actor_names)

    # Update the DBpedia URI in the MovieLens dataset
    for movie_title, dbpedia_uri in batch_data:
        movies_data.loc[movies_data['Title'] == movie_title, 'DBpediaURI'] = dbpedia_uri

    # Calculate progress and elapsed time
    progress = min(end_idx, len(enriched_results))
    elapsed_time = time.time() - start_time
    print(f"Processed {progress}/{len(enriched_results)} rows. Elapsed time: {elapsed_time:.2f} seconds")

# Write the enriched dataset to a CSV file
enriched_data_file = 'movies_enriched_20.csv'  # Replace with the desired output file path
movies_data.to_csv(enriched_data_file, index=False)

# Close the pool of worker processes
pool.close()
pool.join()

Processed 10/20 rows. Elapsed time: 5.19 seconds
Processed 20/20 rows. Elapsed time: 10.37 seconds
Processed 20/20 rows. Elapsed time: 10.37 seconds


checked succeed, run for all rows in movie lens 1M data

In [9]:
import pandas as pd
from SPARQLWrapper import SPARQLWrapper, JSON
from multiprocessing import Pool
import time

def get_director_from_dbpedia(movie_data):
    movie_title, dbpedia_uri = movie_data
    query = f'''
        PREFIX dbo: <http://dbpedia.org/ontology/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        SELECT DISTINCT ?directorName
        WHERE {{
            <{dbpedia_uri}> dbo:director ?director .
            ?director rdfs:label ?directorName .
            FILTER (LANG(?directorName) = 'en')
        }}
    '''

    sparql = SPARQLWrapper('http://dbpedia.org/sparql')
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    bindings = results['results']['bindings']

    if bindings:
        return movie_title, bindings[0]['directorName']['value']
    else:
        return movie_title, None

def get_starring_actors_from_dbpedia(movie_data):
    movie_title, dbpedia_uri = movie_data
    query = f'''
        PREFIX dbo: <http://dbpedia.org/ontology/>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        SELECT DISTINCT ?actorName
        WHERE {{
            <{dbpedia_uri}> dbo:starring ?actor .
            ?actor rdfs:label ?actorName .
            FILTER (LANG(?actorName) = 'en')
        }}
    '''

    sparql = SPARQLWrapper('http://dbpedia.org/sparql')
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    bindings = results['results']['bindings']

    actors = []
    for binding in bindings:
        actors.append(binding['actorName']['value'])

    return movie_title, actors

# Load the MovieLens 1M dataset
movies_file = 'movies.dat'  # Replace with the actual path to your movies file
movies_data = pd.read_csv(movies_file, sep='::', engine='python', names=['MovieID', 'Title', 'Genres'], encoding='latin1')

# Load the mapping file that contains the DBpedia URIs for each MovieLens movie title
mapping_file = 'MappingMovielens2DBpedia-1.2.tsv'  # Replace with the actual path to your mapping file
mapping_data = pd.read_csv(mapping_file, sep='\t', names=['MovieLensTitle', 'DBpediaURI'], encoding='utf-8')

# Create empty columns for directors, starring actors, and DBpedia URIs
movies_data['Director'] = ""
movies_data['Starring'] = ""
movies_data['DBpediaURI'] = ""

# Define the number of worker processes to use
num_processes = 4

# Create a pool of worker processes
pool = Pool(processes=num_processes)

# Enrich the MovieLens dataset with director information, starring actors, and DBpedia URIs
enriched_results = []
for idx, row in movies_data.iterrows():
    movie_title = row['Title']

    # Check if the movie title exists in the mapping dictionary
    if movie_title in mapping_data['MovieLensTitle'].values:
        # Retrieve the corresponding DBpedia URI
        dbpedia_uri = mapping_data.loc[mapping_data['MovieLensTitle'] == movie_title, 'DBpediaURI'].iloc[0]
        # Add the movie data to the list for processing
        enriched_results.append((movie_title, dbpedia_uri))

# Use parallel processing to retrieve director and actor information in batches
batch_size = 10
num_batches = len(enriched_results) // batch_size + 1

start_time = time.time()

for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = (i + 1) * batch_size
    batch_data = enriched_results[start_idx:end_idx]

    # Use parallel processing to retrieve director and actor information
    director_results = pool.map(get_director_from_dbpedia, batch_data)
    actor_results = pool.map(get_starring_actors_from_dbpedia, batch_data)

    # Update the MovieLens dataset with the retrieved information
    for movie_title, director_name in director_results:
        if director_name is not None:
            movies_data.loc[movies_data['Title'] == movie_title, 'Director'] = director_name

    for movie_title, actor_names in actor_results:
        if len(actor_names) >= 3:
            movies_data.loc[movies_data['Title'] == movie_title, 'Starring'] = ', '.join(actor_names)

    # Update the DBpedia URI in the MovieLens dataset
    for movie_title, dbpedia_uri in batch_data:
        movies_data.loc[movies_data['Title'] == movie_title, 'DBpediaURI'] = dbpedia_uri

    # Calculate progress and elapsed time
    progress = min(end_idx, len(enriched_results))
    elapsed_time = time.time() - start_time
    print(f"Processed {progress}/{len(enriched_results)} rows. Elapsed time: {elapsed_time:.2f} seconds")

# Write the enriched dataset to a CSV file
enriched_data_file = 'movies_enriched.csv'  # Replace with the desired output file path
movies_data.to_csv(enriched_data_file, index=False)

# Close the pool of worker processes
pool.close()
pool.join()

Processed 10/3300 rows. Elapsed time: 5.20 seconds
Processed 20/3300 rows. Elapsed time: 10.36 seconds
Processed 30/3300 rows. Elapsed time: 15.54 seconds
Processed 40/3300 rows. Elapsed time: 20.77 seconds
Processed 50/3300 rows. Elapsed time: 25.96 seconds
Processed 60/3300 rows. Elapsed time: 31.15 seconds
Processed 70/3300 rows. Elapsed time: 36.32 seconds
Processed 80/3300 rows. Elapsed time: 41.50 seconds
Processed 90/3300 rows. Elapsed time: 46.85 seconds
Processed 100/3300 rows. Elapsed time: 52.01 seconds
Processed 110/3300 rows. Elapsed time: 57.18 seconds
Processed 120/3300 rows. Elapsed time: 62.34 seconds
Processed 130/3300 rows. Elapsed time: 67.50 seconds
Processed 140/3300 rows. Elapsed time: 72.66 seconds
Processed 150/3300 rows. Elapsed time: 77.81 seconds
Processed 160/3300 rows. Elapsed time: 82.98 seconds
Processed 170/3300 rows. Elapsed time: 88.16 seconds
Processed 180/3300 rows. Elapsed time: 93.41 seconds
Processed 190/3300 rows. Elapsed time: 98.58 seconds
Pro